In [8]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


In [2]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")


Train: (129477, 46)
Val:   (43755, 46)
Test:  (12853, 46)


In [3]:
TARGET = "log_price"
DROP_COLS = ["ClosePrice", "CloseDate", "price_ratio", TARGET]
feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_val, y_val = val_df[feature_cols], val_df[TARGET]
X_test, y_test = test_df[feature_cols], test_df[TARGET]

RANDOM_STATE = 20260805


In [4]:
def evaluate(y_true_log, y_pred_log, label=""):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    mdape = np.median(np.abs((y_true - y_pred) / y_true)) * 100

    print(
        f"{label:>10} | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%  MdAPE: {mdape:.2f}%"
    )
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape, "mdape": mdape}


In [5]:
lower_cap = train_df["price_ratio"].quantile(0.005)
upper_cap = train_df["price_ratio"].quantile(0.9995)
print(f"Lower cap: {lower_cap:.4f}, Upper cap: {upper_cap:.4f}")

train_mask = (train_df["price_ratio"] >= lower_cap) & (
    train_df["price_ratio"] <= upper_cap
)
val_mask = (val_df["price_ratio"] >= lower_cap) & (val_df["price_ratio"] <= upper_cap)
test_mask = (test_df["price_ratio"] >= lower_cap) & (
    test_df["price_ratio"] <= upper_cap
)

print(
    f"Train: {train_mask.sum()} / {len(train_df)} kept ({(1 - train_mask.mean()) * 100:.4f}% removed)"
)
print(
    f"Val: {val_mask.sum()} / {len(val_df)} kept ({(1 - val_mask.mean()) * 100:.4f}% removed)"
)
print(
    f"Test: {test_mask.sum()} / {len(test_df)} kept ({(1 - test_mask.mean()) * 100:.4f}% removed)"
)


Lower cap: 0.7893, Upper cap: 1.9807
Train: 128764 / 129477 kept (0.5507% removed)
Val: 43507 / 43755 kept (0.5668% removed)
Test: 12794 / 12853 kept (0.4590% removed)


In [6]:
X_train_no_outliers = X_train[train_mask]
y_train_no_outliers = y_train[train_mask]

X_val_no_outliers = X_val[val_mask]
y_val_no_outliers = y_val[val_mask]

X_test_no_outliers = X_test[test_mask]
y_test_no_outliers = y_test[test_mask]


In [9]:
dt_tuned = DecisionTreeRegressor(max_depth=7, random_state=RANDOM_STATE)
dt_tuned.fit(X_train_no_outliers, y_train_no_outliers)

rf_tuned = RandomForestRegressor(
    max_depth=None,
    max_features=0.5,
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_tuned.fit(X_train_no_outliers, y_train_no_outliers)

xgb_tuned = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_tuned.fit(X_train_no_outliers, y_train_no_outliers)

lgb_tuned = lgb.LGBMRegressor(
    max_depth=7,
    learning_rate=0.15,
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_tuned.fit(X_train_no_outliers, y_train_no_outliers)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004232 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 13.779382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,7
,learning_rate,0.15
,n_estimators,600
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


## test set metrics

In [10]:
def evaluate(y_true_log, y_pred_log, label=""):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    mdape = np.median(np.abs((y_true - y_pred) / y_true)) * 100
    print(
        f"{label:>10} | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%  MdAPE: {mdape:.2f}%"
    )
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape, "mdape": mdape}


In [11]:
results = {
    "Decision Tree": evaluate(
        y_test_no_outliers, dt_tuned.predict(X_test_no_outliers), "DT"
    ),
    "Random Forest": evaluate(
        y_test_no_outliers, rf_tuned.predict(X_test_no_outliers), "RF"
    ),
    "XGBoost": evaluate(
        y_test_no_outliers, xgb_tuned.predict(X_test_no_outliers), "XGB"
    ),
    "LightGBM": evaluate(
        y_test_no_outliers, lgb_tuned.predict(X_test_no_outliers), "LGBM"
    ),
}
metrics_summary = pd.DataFrame(results).T
metrics_summary.to_csv("metrics_summary.csv")
metrics_summary


        DT | RMSE: $761,310  MAE: $265,105  R2: 0.7286  MAPE: 17.66%  MdAPE: 12.65%
        RF | RMSE: $623,992  MAE: $188,827  R2: 0.8177  MAPE: 11.98%  MdAPE: 7.83%
       XGB | RMSE: $571,068  MAE: $179,497  R2: 0.8473  MAPE: 11.58%  MdAPE: 7.88%
      LGBM | RMSE: $569,733  MAE: $181,408  R2: 0.8480  MAPE: 11.65%  MdAPE: 8.02%


,rmse,mae,r2,mape,mdape
Decision Tree,761309.520090,265105.106279,0.728634,17.659137,12.653723
Random Forest,623992.276252,188826.853504,0.817698,11.976507,7.829490
XGBoost,571067.898968,179496.958732,0.847311,11.579319,7.878584
LightGBM,569733.282754,181407.934273,0.848024,11.647235,8.022843


## Price Band

In [12]:
def price_band_analysis(model, X_test, y_test_log, model_name=""):
    y_true = np.exp(y_test_log)
    y_pred = np.exp(model.predict(X_test))

    bands = pd.cut(
        y_true,
        bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
        labels=["<$500K", "$500K-$1M", "$1M-$2M", "$2M+"],
    )

    results = []
    for band in bands.cat.categories:
        mask = bands == band
        if mask.sum() == 0:
            continue
        yt, yp = y_true[mask], y_pred[mask]
        rmse = root_mean_squared_error(yt, yp)
        mae = mean_absolute_error(yt, yp)
        mape = np.mean(np.abs((yt - yp) / yt)) * 100
        mdape = np.median(np.abs((yt - yp) / yt)) * 100
        results.append(
            {
                "price_band": band,
                "n_listings": mask.sum(),
                "rmse": rmse,
                "mae": mae,
                "mape": mape,
                "mdape": mdape,
            }
        )

    band_df = pd.DataFrame(results)
    print(f"=== {model_name} — Performance by Price Band ===")
    print(band_df.to_string(index=False))
    return band_df


lgb_band_results = price_band_analysis(
    lgb_tuned, X_test_no_outliers, y_test_no_outliers, "LightGBM"
)


=== LightGBM — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        1785 7.799584e+04  51434.454122 15.797741  8.624509
 $500K-$1M        5316 9.739229e+04  66699.302289  8.882893  6.266535
   $1M-$2M        3908 2.246997e+05 162158.488079 11.404992  8.854457
      $2M+        1785 1.477048e+06 695144.956917 16.259711 13.119755


# Week 8 — Evaluation Expansion: Summary

added MdAPE alongside MAPE to get a fuller picture of prediction error, and broke down LightGBM's performance by price band.

**Overall model comparison:** LightGBM and XGBoost remain the top performers (R² 0.848 and 0.847), followed by Random Forest (0.818) and Decision Tree (0.729). Across all models, MAPE is noticeably higher than MdAPE — largest for Decision Tree (17.66% vs. 12.65%) — showing that a small number of hard-to-predict listings pull up the mean error while most predictions are actually quite accurate.

**Price band analysis (LightGBM):** The model performs best in the $500K–$1M range (MAPE 8.88%), which also has the most listings. Accuracy drops sharply above $2M — RMSE jumps 15x and MAPE nearly doubles to 16.26% — consistent with luxury homes being underrepresented in the data and having more unique, hard-to-capture features. Homes under $500K also show a higher MAPE (15.80%), but this is a scale effect: absolute errors are actually the smallest in this band, they just represent a larger percentage of a lower price base.

**Bottom line:** The model is most reliable in the mid-market ($500K–$1M) and least reliable at the luxury end ($2M+), for structural reasons — data sparsity and property uniqueness — rather than a fixable modeling flaw.